# 36 - Benchmark: JAFFE Dataset

**Dataset:** JAFFE — 213 gambar, 7 emosi, 10 subjek (wanita Jepang)
**Evaluasi:** Single Split (80/10/10 subject-wise)
**Model:** Sama dengan eksperimen utama (CNN, FCNN, Late Fusion, Intermediate, + TL)
**Skenario:** B1 (Baseline) saja — dataset JAFFE sudah seimbang (~30/kelas)
**Konfigurasi kelas:** 7-class dan 4-class

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

BATCH_SIZE = 16  # smaller for small dataset
EPOCHS = 50
PATIENCE = 15
OUTPUT_DIR = PROJECT_ROOT / "models" / "benchmark" / "jaffe"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model configs: (name, class, model_type, lr)
MODELS = [
    ("CNN", EmotionCNN, "cnn", 0.0001),
    ("FCNN", EmotionFCNN, "fcnn", 0.0001),
    ("Intermediate", IntermediateFusion, "fusion", 0.0001),
    ("CNN_TL", EmotionCNNTransfer, "cnn", 0.00005),
    ("Intermediate_TL", IntermediateFusionTransfer, "fusion", 0.00005),
]

print("Setup complete.")

Device: cuda
GPU: Tesla T4
Setup complete.


In [2]:
# ── Helper Functions ──

def make_loader(images, landmarks, labels, model_type, batch_size=16, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == "cnn": ds = TensorDataset(img_t, y_t)
    elif model_type == "fcnn": ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def subject_split(subjects, seed=42, train_ratio=0.8, val_ratio=0.1):
    rng = np.random.RandomState(seed)
    unique = np.array(sorted(set(subjects)))
    rng.shuffle(unique)
    n = len(unique)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    return unique[:n_train], unique[n_train:n_train+n_val], unique[n_train+n_val:]


def get_split_indices(subjects, subject_indices, train_subjs, val_subjs, test_subjs):
    train_idx = np.concatenate([subject_indices[s] for s in train_subjs]) if len(train_subjs) > 0 else np.array([], dtype=int)
    val_idx = np.concatenate([subject_indices[s] for s in val_subjs]) if len(val_subjs) > 0 else np.array([], dtype=int)
    test_idx = np.concatenate([subject_indices[s] for s in test_subjs]) if len(test_subjs) > 0 else np.array([], dtype=int)
    return train_idx, val_idx, test_idx


def train_and_eval(ModelClass, model_type, lr,
                   train_img, train_lm, train_y,
                   val_img, val_lm, val_y,
                   test_img, test_lm, test_y,
                   emotions, save_dir):
    tr_loader = make_loader(train_img, train_lm, train_y, model_type, BATCH_SIZE)
    val_loader = make_loader(val_img, val_lm, val_y, model_type, BATCH_SIZE, False)
    test_loader = make_loader(test_img, test_lm, test_y, model_type, BATCH_SIZE, False)

    model = ModelClass(num_classes=len(emotions)).to(device)
    save_path = str(save_dir / "model.pth")
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-7)

    train_model(model, tr_loader, val_loader, criterion, optimizer, scheduler,
                device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, model_type, emotions)
    return {"accuracy": float(r["test_accuracy"]),
            "macro_f1": float(r["test_macro_f1"]),
            "weighted_f1": float(r["test_weighted_f1"])}


def late_fusion_eval(train_img, train_lm, train_y,
                     val_img, val_lm, val_y,
                     test_img, test_lm, test_y,
                     num_classes, save_dir):
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    tr_cnn = make_loader(train_img, train_lm, train_y, "cnn", BATCH_SIZE)
    val_cnn = make_loader(val_img, val_lm, val_y, "cnn", BATCH_SIZE, False)
    opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(cnn, tr_cnn, val_cnn, nn.CrossEntropyLoss(), opt, sch,
                device, "cnn", EPOCHS, PATIENCE, str(save_dir / "cnn.pth"))

    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    tr_fcnn = make_loader(train_img, train_lm, train_y, "fcnn", BATCH_SIZE)
    val_fcnn = make_loader(val_img, val_lm, val_y, "fcnn", BATCH_SIZE, False)
    opt2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
    sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(fcnn, tr_fcnn, val_fcnn, nn.CrossEntropyLoss(), opt2, sch2,
                device, "fcnn", EPOCHS, PATIENCE, str(save_dir / "fcnn.pth"))

    cnn.load_state_dict(torch.load(save_dir / "cnn.pth", map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(save_dir / "fcnn.pth", map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()

    test_img_t = torch.from_numpy(test_img).permute(0, 3, 1, 2).to(device)
    test_lm_t = torch.from_numpy(test_lm).to(device)
    with torch.no_grad():
        cnn_probs = torch.softmax(cnn(test_img_t), dim=1).cpu().numpy()
        fcnn_probs = torch.softmax(fcnn(test_lm_t), dim=1).cpu().numpy()

    best_f1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w * cnn_probs + (1-w) * fcnn_probs).argmax(axis=1)
        f1 = f1_score(test_y, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1 = f1; best_w = w; best_preds = preds

    acc = accuracy_score(test_y, best_preds)
    wf1 = f1_score(test_y, best_preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "macro_f1": best_f1, "weighted_f1": wf1, "best_cnn_weight": best_w}


def run_benchmark(dataset_name, data_dir, num_classes, emotions):
    """Run all models B1 only with single split on benchmark dataset."""
    print(f"\n{'='*70}")
    print(f"  BENCHMARK: {dataset_name} ({num_classes}-class, Single Split, B1 only)")
    print(f"{'='*70}")

    images = np.load(data_dir / "X_images.npy")
    landmarks = np.load(data_dir / "X_landmarks.npy")
    labels = np.load(data_dir / "y_labels.npy")
    subjects = np.load(data_dir / "subjects.npy", allow_pickle=True)

    unique_subjects = sorted(set(subjects))
    subject_indices = {s: np.where(subjects == s)[0] for s in unique_subjects}

    train_subjs, val_subjs, test_subjs = subject_split(subjects)
    train_idx, val_idx, test_idx = get_split_indices(subjects, subject_indices, train_subjs, val_subjs, test_subjs)

    print(f"  Total: {len(labels)} samples, {len(unique_subjects)} subjects")
    print(f"  Train: {len(train_idx)} ({len(train_subjs)} subj) | Val: {len(val_idx)} ({len(val_subjs)} subj) | Test: {len(test_idx)} ({len(test_subjs)} subj)")

    all_results = {}
    models_to_run = MODELS + [("Late_Fusion", None, "late", 0.0001)]

    for model_name, ModelClass, model_type, lr in models_to_run:
        key = f"{model_name}_B1"
        print(f"\n  >> {key}...", end=" ")

        save_dir = OUTPUT_DIR / f"{dataset_name}_{num_classes}c" / key
        os.makedirs(save_dir, exist_ok=True)

        tr_img, tr_lm, tr_y = images[train_idx], landmarks[train_idx], labels[train_idx]
        v_img, v_lm, v_y = images[val_idx], landmarks[val_idx], labels[val_idx]
        te_img, te_lm, te_y = images[test_idx], landmarks[test_idx], labels[test_idx]

        if model_type == "late":
            r = late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                  te_img, te_lm, te_y, num_classes, save_dir)
        else:
            r = train_and_eval(ModelClass, model_type, lr,
                                tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                te_img, te_lm, te_y, emotions, save_dir)

        all_results[key] = r
        print(f"Acc={r['accuracy']:.4f} F1={r['macro_f1']:.4f}")

    save_path = OUTPUT_DIR / f"{dataset_name}_{num_classes}c_results.json"
    with open(save_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\n  Saved: {save_path}")
    return all_results

print("Helper functions ready.")

Helper functions ready.


## Run JAFFE Benchmark

In [3]:
BENCHMARK_DIR = PROJECT_ROOT / "data" / "benchmark"
EMOTIONS_7 = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]
EMOTIONS_4 = ["neutral", "happy", "sad", "negative"]

# 7-class JAFFE
res_7c = run_benchmark("jaffe", BENCHMARK_DIR / "jaffe_7class", 7, EMOTIONS_7)

# 4-class JAFFE
res_4c = run_benchmark("jaffe", BENCHMARK_DIR / "jaffe_4class", 4, EMOTIONS_4)


  BENCHMARK: jaffe (7-class, Single Split, B1 only)


  Total: 213 samples, 10 subjects
  Train: 173 (8 subj) | Val: 20 (1 subj) | Test: 20 (1 subj)

  >> CNN_B1... 

 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      2.0598     0.1445     1.9855    0.1500   0.0373   0.000100  (2.1s)


     2      1.9748     0.2139     2.0287    0.1500   0.0373   0.000100  (1.2s)


...

    23      0.8456     0.7052     1.1558    0.6500   0.5385   0.000100  (0.1s)
    24      0.7891     0.6936     1.1450    0.6000   0.4582   0.000100  (0.1s)
    25      0.7988     0.6994     1.2135    0.5000   0.4381   0.000100  (0.1s)
    26      0.7969     0.6994     1.2528    0.4500   0.3583   0.000100  (0.1s)


    27      0.8083     0.7168     1.1593    0.5000   0.3861   0.000100  (0.0s)
    28      0.7716     0.6994     1.2212    0.4500   0.4685   0.000100  (0.1s)
    29      0.7449     0.7457     1.2166    0.4000   0.2167   0.000100  (0.1s)
    30      0.8070     0.6821     1.2233    0.5500   0.3071   0.000100  (0.1s)


    31      0.6823     0.7399     1.2119    0.4000   0.2381   0.000100  (0.1s)
    32      0.6847     0.7572     1.2715    0.4500   0.4048   0.000100  (0.1s)
    33      0.7045     0.7630     1.3392    0.3000   0.3255   0.000050  (0.0s)
    34      0.7177     0.7688     1.2766    0.3500   0.3750   0.000050  (0.1s)


    35      0.7237     0.7168     1.2422    0.4500   0.5167   0.000050  (0.1s)
    36      0.6900     0.7861     1.3055    0.3500   0.3542   0.000050  (0.0s)
    37      0.6681     0.7514     1.2489    0.3500   0.3542   0.000050  (0.0s)
    38      0.6405     0.7688     1.2780    0.3500   0.2039   0.000050  (0.0s)

Early stopping at epoch 38. Best epoch: 23 (val_f1=0.5385)

Best: epoch 23, val_acc=0.6500, val_f1=0.5385
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe/jaffe_4c/Late_Fusion_B1/fcnn.pth


Acc=0.6500 F1=0.3964

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe/jaffe_4c_results.json


## Ringkasan JAFFE

In [4]:
# Summary table
for nc_label, res in [("7-class", res_7c), ("4-class", res_4c)]:
    print(f"\n{'='*70}")
    print(f"  JAFFE {nc_label} - Single Split B1 Results (sorted by Macro F1)")
    print(f"{'='*70}")
    print(f"  {'Model':<30} {'Macro F1':>10} {'Accuracy':>10} {'W-F1':>10}")
    print(f"  {'-'*62}")
    for key in sorted(res.keys(), key=lambda k: -res[k]["macro_f1"]):
        r = res[key]
        print(f"  {key:<30} {r['macro_f1']:>10.4f} {r['accuracy']:>10.4f} {r['weighted_f1']:>10.4f}")

# Compare with own dataset
print(f"\n{'='*70}")
print(f"  PERBANDINGAN: JAFFE vs Dataset Sendiri (Front-Only, B1)")
print(f"{'='*70}")
print(f"  {'Model':<25} {'JAFFE 7c':>10} {'Own 7c':>10} {'JAFFE 4c':>10} {'Own 4c':>10}")
print(f"  {'-'*58}")

own_results = {
    "CNN_B1": {"7c": 0.137, "4c": 0.238},
    "FCNN_B1": {"7c": 0.158, "4c": 0.307},
    "Intermediate_B1": {"7c": 0.137, "4c": 0.243},
    "CNN_TL_B1": {"7c": 0.154, "4c": 0.274},
    "Intermediate_TL_B1": {"7c": 0.173, "4c": 0.412},
}

for model_key, own in own_results.items():
    j7 = res_7c.get(model_key, {}).get("macro_f1", 0)
    j4 = res_4c.get(model_key, {}).get("macro_f1", 0)
    print(f"  {model_key:<25} {j7:>10.4f} {own['7c']:>10.4f} {j4:>10.4f} {own['4c']:>10.4f}")


  JAFFE 7-class - Single Split B1 Results (sorted by Macro F1)
  Model                            Macro F1   Accuracy       W-F1
  --------------------------------------------------------------
  Late_Fusion_B1                     0.5452     0.6000     0.5225
  CNN_TL_B1                          0.4639     0.5000     0.4371
  Intermediate_TL_B1                 0.4473     0.4500     0.4197
  CNN_B1                             0.3040     0.4500     0.3192
  FCNN_B1                            0.2088     0.2500     0.1692
  Intermediate_B1                    0.0373     0.1500     0.0391

  JAFFE 4-class - Single Split B1 Results (sorted by Macro F1)
  Model                            Macro F1   Accuracy       W-F1
  --------------------------------------------------------------
  FCNN_B1                            0.4381     0.5500     0.5295
  Late_Fusion_B1                     0.3964     0.6500     0.5521
  Intermediate_TL_B1                 0.3750     0.6500     0.5583
  CNN_TL_B1     